# Median Beat

Build a single representative beat from one or many ECG recordings.

The pipeline has three stages:

| Stage | What it does | Where |
|---|---|---|
| 0. Harmonize | raw signal -> common sampling rate, length, lead order (mV) | `harmonize` |
| 1. Per-ECG beat | R-peak detection -> one median beat per recording | `RlignMedianBeat.per_ecg_beats` |
| 2. Aggregate | one beat per patient -> robust median across the cohort | `MedianBeatExtractor.aggregate_beats` |

`build()` runs stages 1-2 for you.

> **Kernel**: run this with the environment that has ExECG installed:
> `pip install -e .` includes neurokit2 and scikit-learn.

In [1]:
import json

import matplotlib.pyplot as plt
import numpy as np

from execg.median_beat import (
    RlignMedianBeat,
    assemble_leads,
    harmonize,
    LEAD_NAMES_12,
)
from execg.visualizer import plot_median_beats

## 1. Load a sample ECG

In [2]:
with open("../samples/data/sample.json") as f:
    sample = json.load(f)

fs = 500  # the bundled sample is 10 s at 500 Hz

# assemble_leads stacks a {lead: samples} dict into the standard 12-lead order,
# deriving III/aVR/aVL/aVF from I and II when they are missing.
raw, present = assemble_leads(sample["data"], LEAD_NAMES_12)
print(f"raw: {raw.shape}, {fs} Hz, leads present: {len(present)}/12")

raw: (12, 5000), 500 Hz, leads present: 12/12


## 2. Harmonize

`harmonize` resamples to a common rate and crops/pads to a fixed duration. Every
extractor expects its input already in this form.

In [3]:
signal = harmonize(raw, fs_in=fs, target_fs=500, seconds=10)
print(f"harmonized: {signal.shape}")  # (12, 5000)

# build() takes a batch: (n_ecgs, n_leads, length). One recording -> add an axis.
signals = signal[None]

harmonized: (12, 5000)


## 3. Traditional median beat

`scale_method="none"` leaves the time axis alone: beats are cut in a fixed window
around each R-peak and median-averaged. This is the classic median beat, directly
comparable to what ECG carts (GE 12SL, UNI-G) produce.

In [4]:
traditional = RlignMedianBeat(
    scale_method="none",  # no heart-rate correction
    beat_window_ms=(400, 800),  # 400 ms before / 800 ms after the R-peak
    num_workers=1,
    verbose=False,
)

beat = traditional.build(signals, label="traditional")
print(f"beat: {beat.beat.shape}  ({traditional.beat_length} samples = 1.2 s at 500 Hz)")
print(f"template_bpm: {beat.template_bpm}  (None - the time axis is not warped)")
print(f"n_ecgs={beat.n_ecgs}, n_dropped={beat.n_dropped}")

fig, _ = plot_median_beats({"traditional": beat}, sampling_rate=beat.sampling_rate)
plt.show()

beat: (12, 600)  (600 samples = 1.2 s at 500 Hz)
template_bpm: None  (None - the time axis is not warped)
n_ecgs=1, n_dropped=0


## 4. Heart-rate corrected median beat

`scale_method="hrc"` warps every beat onto a common `template_bpm`, keeping the
QRS core rate-independent while rescaling the ST-T region per the QT-RR relation.
Use it when you pool recordings whose **heart rates differ**, so T-wave position
is not confounded by rate.

In [5]:
hrc = RlignMedianBeat(scale_method="hrc", template_bpm=60, num_workers=1, verbose=False)

beat_hrc = hrc.build(signals, label="hrc")
print(f"beat: {beat_hrc.beat.shape}  (60 bpm -> {hrc.beat_length} samples = 1.0 s)")
print(f"template_bpm: {beat_hrc.template_bpm}")

beat: (12, 500)  (60 bpm -> 500 samples = 1.0 s)
template_bpm: 60


## 5. Comparing the scale methods

The three methods produce different time axes, so the curves are **not** aligned
sample-by-sample - compare the morphology, not the absolute timing.

In [6]:
methods = {}
for name in ["none", "linear", "hrc"]:
    extractor = RlignMedianBeat(
        scale_method=name, template_bpm=60, num_workers=1, verbose=False
    )
    methods[name] = extractor.build(signals, label=name)
    print(f"{name:7s} -> {methods[name].beat.shape}")

fig, _ = plot_median_beats(
    methods,
    sampling_rate=500,
    leads=["II", "V2", "V5"],  # a subset keeps the figure readable
    ncols=3,
    title="scale_method comparison",
)
plt.show()

none    -> (12, 600)
linear  -> (12, 500)
hrc     -> (12, 500)


## 6. Cohorts: many recordings, many patients

With a real cohort you pass `(n_ecgs, 12, length)` plus `subject_ids`. Two options
matter here:

- `per_patient=True` collapses each patient to one beat first, so a patient with
  many recordings cannot dominate the group.
- `keep_per_beat=True` retains the individual beats, which lets the plot shade a
  confidence band with `show_band=True`.

In [7]:
# Stand in for a cohort by jittering the sample recording.
rng = np.random.default_rng(0)
cohort = np.stack(
    [
        signal * s + rng.normal(0, 0.01, signal.shape)
        for s in [0.9, 0.95, 1.0, 1.05, 1.1, 1.15]
    ]
)
subject_ids = ["p1", "p1", "p2", "p2", "p3", "p3"]

extractor = RlignMedianBeat(
    scale_method="none",
    per_patient=True,  # one vote per patient
    keep_per_beat=True,  # keep contributions so we can draw a band
    num_workers=1,
    verbose=False,
)

cohort_beat = extractor.build(cohort, subject_ids=subject_ids, label="cohort")
print(
    f"n_ecgs={cohort_beat.n_ecgs}, n_patients={cohort_beat.n_patients}, "
    f"n_dropped={cohort_beat.n_dropped}"
)

fig, _ = plot_median_beats(
    {"cohort": cohort_beat},
    sampling_rate=cohort_beat.sampling_rate,
    leads=["II", "V2", "V5"],
    ncols=3,
    show_band=True,  # 95% band across the contributing beats
    title="Cohort median beat with 95% band",
)
plt.show()

n_ecgs=6, n_patients=3, n_dropped=0


## 7. Comparing groups

`build_groups` runs `build` per named group and returns a dict the plotter takes
directly - this is the usual way to contrast cohorts (e.g. by serum potassium).

In [8]:
groups = {"low": cohort[:3] * 0.8, "high": cohort[3:] * 1.2}
group_sids = {"low": ["p1", "p1", "p2"], "high": ["p2", "p3", "p3"]}

group_beats = extractor.build_groups(groups, subject_ids=group_sids)
for label, b in group_beats.items():
    print(f"{label:5s} n_ecgs={b.n_ecgs} n_patients={b.n_patients}")

fig, _ = plot_median_beats(
    group_beats,
    sampling_rate=500,
    leads=["II", "V2", "V5"],
    ncols=3,
    title="Group comparison",
)
plt.show()

low   n_ecgs=3 n_patients=2
high  n_ecgs=3 n_patients=2


## 8. Adding your own algorithm

Every algorithm implements `MedianBeatExtractor` and only has to supply **stage 1**
(`per_ecg_beats`). Harmonization, per-patient reduction, robust aggregation,
clustering and `build_groups` are all inherited.

In [9]:
from execg.median_beat import MedianBeatExtractor


class FixedWindowMedianBeat(MedianBeatExtractor):
    """Cut a fixed window at the largest deflection in the R-peak lead."""

    def per_ecg_beats(self, signals):
        beat_len = 600
        out = np.full((signals.shape[0], signals.shape[1], beat_len), np.nan)
        for i, ecg in enumerate(signals):
            if ecg.shape[1] < beat_len:
                continue  # stays NaN -> dropped
            peak = int(np.argmax(np.abs(ecg[1])))  # lead II
            start = int(np.clip(peak - 200, 0, ecg.shape[1] - beat_len))
            out[i] = ecg[:, start : start + beat_len]
        return out


custom = FixedWindowMedianBeat(verbose=False).build(cohort, subject_ids=subject_ids)
print(f"custom beat: {custom.beat.shape}, n_patients={custom.n_patients}")

custom beat: (12, 600), n_patients=3
